In [ ]:
# ========== 第 2 天任务：抓取电商页 → 用本地 Ollama（llama3.2）抽取商品 ==========
# 练习目标：Day1 同一套抓取 + messages 流程，把后端从云端 GPT 换成本地 Llama（OpenAI 兼容 /v1）
# 怎么跑：先启动 Ollama 并 pull llama3.2；本机 11434 可访问后，从上到下运行；末尾可改 URL / 商品名

# 导入标准库 os：读环境变量（本练习主要用本地端点，仍保留常见导入习惯）
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进环境（若有其它配置可一并加载）
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮展示模型返回的 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：通过改 base_url 指向本地 Ollama 的 OpenAI 兼容 API
from openai import OpenAI
# 从 bs4 导入 BeautifulSoup：解析 HTML，去掉无关标签后抽出纯文本
from bs4 import BeautifulSoup
# 导入 requests：用 HTTP GET 下载目标网页
import requests


# 浏览器风格 User-Agent：很多网站会拒无头爬虫；字符串保持原样
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# 加载 .env 到环境变量
load_dotenv()
# 仍创建默认 OpenAI 客户端（本格 summarize 走 ollama；保留原代码结构）
openai = OpenAI()



# Ollama 的 OpenAI 兼容 Base URL（/v1）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 指向本地 Ollama；api_key='ollama' 为占位密钥（本地端点通常不校验）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# system prompt：角色 + 抽取规则 + Markdown 表格格式；必须保留英文原文
system_prompt = """You are an AI product extraction assistant.

Your task is to analyze raw scraped e-commerce website content and extract structured product listings that match a specified product name provided by the user.

You will be given:

1. The scraped website content (HTML or text)
2. A target product name

Instructions:

* Identify all products in the content that match or are closely related to the target product name.

* Ignore text that may be navigation related, including menus, headers, footers, category links, breadcrumbs, filters, login sections, advertisements, or pagination elements.

* Focus only on actual product listing information.

* Extract at least 10 relevant product listings where available.

* For each product extract:

  * Product Name
  * Product Description
  * Product Price

Data Handling Rules:

* Ignore listings that do not contain a visible price.
* Normalize all prices into numeric values (remove currency symbols).
* If a price is given as a range, use the higher value.
* Ignore advertisements or unrelated products.

Sorting Rules:
* Sort the final list strictly on price and quality.


Output Format:

Respond in markdown.
Do not wrap the markdown in a code block - respond just with the markdown.

Return the result as a Markdown table with the following columns:

| Product Name | Description | Price |

Order in ascending order with the lowest price at the top
Provide a summary after the table of the most recomended as per the price.

* The table must contain at least 10 products where available.
* Prices must be shown as numeric values with currency.
* Do not include any commentary before or after the table.
 """
# user prompt 前缀：后面拼接网页正文 + 商品名；英文原文保留
user_prompt = """
 Here is the website content and the product name

"""
def fetch_website_contents(url):
    """抓取 url 对应页面，清洗 HTML 后返回「标题 + 正文」截断文本（最多约 2000 字符）。"""
    # GET 目标页；带上 headers 降低被拒概率
    response = requests.get(url, headers=headers)
    # 用 html.parser 解析响应字节为 DOM
    soup = BeautifulSoup(response.content, "html.parser")
    # 取 <title> 文本；没有标题就用占位英文串
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # 删掉 script/style/img/input 等对摘要无用的节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 从 body 抽出可见文本
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    # 标题 + 正文，截断到 2000 字符，控制本地模型上下文长度
    return (title + "\n\n" + text)[:2_000]

def messages_for(website, product_name):
    """组装 messages：system 定规则，user 放网页内容 + 商品名。"""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website + product_name}
    ]


def summarize(url, product_name):
    """抓取网页 → 调本地 llama3.2（经 ollama 客户端）→ 返回 Markdown 字符串。"""
    website = fetch_website_contents(url)
    # 注意：这里用的是 ollama 客户端，不是上面的 openai；model id 保持 llama3.2
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website, product_name)
    )
    return response.choices[0].message.content

def display_summary(url, product_name):
    """调用 summarize，并在笔记本里用 Markdown 渲染结果。"""
    summary = summarize(url, product_name)
    display(Markdown(summary))    


# 入口示例：抓 amazon.com，目标商品名为 static bikes（与 Day1 对照同一任务、不同后端）
display_summary("https://amazon.com", "static bikes")
